In [1]:
import pandas as pd
import numpy as np
import json
import os
from pathlib import Path


In [2]:
# YouTube category_id → category_name mapping
category_dict = {
    1:  "Film & Animation",
    2:  "Autos & Vehicles",
    10: "Music",
    15: "Pets & Animals",
    17: "Sports",
    18: "Short Movies",
    19: "Travel & Events",
    20: "Gaming",
    21: "Videoblogging",
    22: "People & Blogs",
    23: "Comedy",
    24: "Entertainment",
    25: "News & Politics",
    26: "Howto & Style",
    27: "Education",
    28: "Science & Technology",
    29: "Nonprofits & Activism",
    30: "Movies",
    31: "Anime/Animation",
    32: "Action/Adventure",
    33: "Classics",
    34: "Comedy",        # duplicate label in YT taxonomy
    35: "Documentary",
    36: "Drama",
    37: "Family",
    38: "Foreign",
    39: "Horror",
    40: "Sci-Fi/Fantasy",
    41: "Thriller",
    42: "Shorts",
    43: "Shows",
    44: "Trailers",
}
print(f"Loaded {len(category_dict)} category mappings.")


Loaded 32 category mappings.


In [11]:
def process_country(file_path: str, verified_path: str, country_code: str) -> pd.DataFrame:
    """
    Process one country's YouTube trending CSV.

    Parameters
    ----------
    file_path     : path to the country CSV  (e.g. "USvideos.csv")
    verified_path : path to the verified-embeds CSV for that country
    country_code  : two-letter code, e.g. "US"

    Returns
    -------
    Processed DataFrame ready to append to the master file.
    """

    # ── 1. Load & initial column selection ───────────────────────────────────
    KEEP_COLS = [
        "video_id", "trending_date", "title", "channel_title",
        "category_id", "publish_time", "tags", "views", "likes",
        "dislikes", "comment_count", "thumbnail_link"
    ]

    df = pd.read_csv(
        file_path,
        usecols=KEEP_COLS,
        on_bad_lines="skip",
        encoding="utf-8",
        encoding_errors="replace",
    )
    print(f"[{country_code}] Loaded {len(df):,} rows from {Path(file_path).name}")

    # ── 2. Deduplication & trending_frequency ────────────────────────────────
    # Count how many times each video appeared in the trending list
    freq = df.groupby("video_id").size().rename("trending_frequency")
    df = df.merge(freq, on="video_id")

    # Sort so the LAST (most recent) row wins when we drop duplicates
    df = df.sort_values("trending_date")
    df = df.drop_duplicates(subset="video_id", keep="last").reset_index(drop=True)
    print(f"[{country_code}] After deduplication: {len(df):,} unique videos")

    # ── 3. Keep only verified video IDs ──────────────────────────────────────
    verified_ids = set()
    if verified_path and Path(verified_path).exists():
        v_df = pd.read_csv(verified_path, usecols=["Verified_Video_ID"] )
        verified_ids = set(v_df["Verified_Video_ID"].dropna().astype(str))
        print(f"[{country_code}] Loaded {len(verified_ids):,} verified embed IDs")
    else:
        print(f"[{country_code}] WARNING: verified embeds file not found → no rows will be kept")

    df = df[df["video_id"].astype(str).isin(verified_ids)].reset_index(drop=True)
    print(f"[{country_code}] After verified filter: {len(df):,} videos")

    # ── 4. Category mapping (category_name only) ─────────────────────────────
    df["category_id"] = pd.to_numeric(df["category_id"], errors="coerce")
    df["category_name"] = df["category_id"].map(category_dict).fillna("Unknown")
    df = df.drop(columns=["category_id"] )

    # ── 5. Velocity score ─────────────────────────────────────────────────────
    views    = df["views"].clip(lower=0)
    likes    = df["likes"].clip(lower=0)
    dislikes = df["dislikes"].clip(lower=0)
    comments = df["comment_count"].clip(lower=0)

    engagement_rate = (likes + comments) / views.replace(0, np.nan)
    sentiment_bias  = likes / (likes + dislikes).replace(0, np.nan)
    raw_velocity    = (
        df["trending_frequency"]
        * np.log10(views + 1)
        * engagement_rate
        * sentiment_bias
    )

    # Normalise 0–100 within the country dataset
    v_min, v_max = raw_velocity.min(), raw_velocity.max()
    if v_max > v_min:
        df["velocity_score"] = (raw_velocity - v_min) / (v_max - v_min) * 100
    else:
        df["velocity_score"] = 0.0
    df["velocity_score"] = df["velocity_score"].fillna(0).round(4)

    # ── 6. Verified-embed flag ────────────────────────────────────────────────
    df["is_verified"] = True

    # ── 7. Mega-string for BGE-M3 ─────────────────────────────────────────────
    def clean_tags(raw):
        if not isinstance(raw, str):
            return ""
        return (
            raw.replace("[", "")
               .replace("]", "")
               .replace('"', "")
               .replace("|", ", ")
               .strip()
        )

    def clean_desc(raw):
        if not isinstance(raw, str):
            return ""
        cleaned = raw.replace("\n", " ").replace("\r", " ")
        return cleaned[:300].strip()

    df["mega_string"] = (
        "Title: "    + df["title"].fillna("").astype(str)
        + " | Category: " + df["category_name"].fillna("").astype(str)
        + " | Tags: "     + df["tags"].apply(clean_tags)
    )

    # ── 8. Country code ───────────────────────────────────────────────────────
    df["country_code"] = country_code

    print(f"[{country_code}] Processing complete. Final shape: {df.shape}")
    return df


In [30]:
# ── CONFIGURE HERE ────────────────────────────────────────────────────────────
# Edit these three variables and run the cell.

COUNTRY_CODE = "RU".upper()  # Change only this value

FILE_PATH = f"./archive/{COUNTRY_CODE}videos.csv"
VERIFIED_PATH = f"verified_embeds_{COUNTRY_CODE}.csv"
# ─────────────────────────────────────────────────────────────────────────────

df_result = process_country(FILE_PATH, VERIFIED_PATH, COUNTRY_CODE)
df_result.head(3)


[RU] Loaded 40,739 rows from RUvideos.csv
[RU] After deduplication: 34,282 unique videos
[RU] Loaded 2,509 verified embed IDs
[RU] After verified filter: 2,509 videos
[RU] Processing complete. Final shape: (2509, 17)


,video_id,trending_date,title,channel_title,publish_time,tags,views,likes,dislikes,comment_count,thumbnail_link,trending_frequency,category_name,velocity_score,is_verified,mega_string,country_code
0,nElxOsHbCZM,17.01.12,Полное интервью Басты для вМесте,вМесте,2017-11-30T15:23:04.000Z,"вМесте|""проект вМесте""|""Союз Охраны Психическо...",3985,176,3,11,https://i.ytimg.com/vi/nElxOsHbCZM/default.jpg,1,People & Blogs,4.6309,True,Title: Полное интервью Басты для вМесте | Cate...,RU
1,JptWgiHG89o,17.01.12,ИНТУИЦИЯ С ОСОБЫМИ ПАКАМИ VS. ДЕНЧИК ФЛОМАСТЕРОВ,Finikland,2017-11-30T12:00:01.000Z,"Fifa|""fut""|""ФИФЕРЫ""|""KEFIR""|""PANDAFX""|""ФИФЕР""|...",86300,6746,250,159,https://i.ytimg.com/vi/JptWgiHG89o/default.jpg,1,Gaming,10.6158,True,Title: ИНТУИЦИЯ С ОСОБЫМИ ПАКАМИ VS. ДЕНЧИК ФЛ...,RU
2,Hwpmjd33EHw,17.01.12,Ветеринарный осмотр ушастой совы с травмой крыла.,Yoll,2017-11-30T15:00:02.000Z,"сова|""ушастая сова""|""птица""|""реабилитация птиц...",14538,1570,4,201,https://i.ytimg.com/vi/Hwpmjd33EHw/default.jpg,1,Pets & Animals,14.0991,True,Title: Ветеринарный осмотр ушастой совы с трав...,RU


In [31]:
MASTER_FILE = "processed_youtube_global.csv"

master_exists = Path(MASTER_FILE).exists()

df_result.to_csv(
    MASTER_FILE,
    mode="a",           # append
    header=not master_exists,
    index=False,
    encoding="utf-8",
)

# Quick sanity check
master_df = pd.read_csv(MASTER_FILE)
counts = master_df["country_code"].value_counts()

print(f"\n✅  Appended {len(df_result):,} rows for [{COUNTRY_CODE}] → {MASTER_FILE}")
print(f"\nMaster file now contains {len(master_df):,} rows across {counts.shape[0]} country/countries:")
print(counts.to_string())



✅  Appended 2,509 rows for [RU] → processed_youtube_global.csv

Master file now contains 24,499 rows across 10 country/countries:
country_code
JP    2658
DE    2579
KR    2557
FR    2514
RU    2509
IN    2508
MX    2501
US    2440
CA    2398
GB    1835
